# 02 — パラメータ探索・設定選択・モデル比較

## 1. Overview

**01_data → CLI Smoke / Validation探索 → 02で設定選択 → 03で推薦を確認 → 設定固定 → CLI最終Test → 02で比較**

- このNotebookは検証ワークフローの中心。既定は読込みだけで、学習・長時間の探索はCLIへ委譲する。
- 現在の状態・次の操作はSetup直下に表示。失敗・条件不一致なら表示された戻り先で確認する。
- [01_data.ipynb](01_data.ipynb)で品質・正解数を確認し、選択後は[03_recommendation.ipynb](03_recommendation.ipynb)へ。
- 探索が未実行の場合、従来の合成保存結果を「参考結果」として表示する。実データや探索済みの最良設定とは区別する。


`experiment.protocol="listening-v2"`では同じ導線で回数・日数に基づく評価を扱います。期間・重み・タスク・主指標は共通TOMLで事前固定し、Testを調整に使いません。

## 2. Setup — Experiment Setupはここだけ

- `MUSIC_RECSYS_EXPERIMENT_CONFIG`で実験TOMLを指定。既定は`configs/workflow-toy.toml`の合成例。
- データ・Snapshot / Splitの出典、モデル・固定パラメータ、探索範囲、seed、主指標、予算は同じTOMLをCLIと共有する。
- `ACTION=None`は読込みのみ。`"select"`または`"freeze"`を明示した場合だけ対応する共通操作を実行する。探索・最終TestはCLIで実行する。
- `REPORT_PATHS`・`MLFLOW_RUNS`以下は既存の保存結果を読むための参考表示設定。探索のモデル設定を二重に持たない。


In [ ]:
import os
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from music_recsys.evaluation.tracking import TrackingConfig
from music_recsys.experiment_analysis import (
    BootstrapConfig,
    additional_metric_rows,
    condition_rows,
    efficiency_rows,
    failure_rows,
    group_rows,
    load_reports,
    paired_differences,
    performance_rows,
    read_mlflow_report,
    repetition_rows,
    user_rows,
)
from music_recsys.experiment_search import (
    context as experiment_context,
)
from music_recsys.experiment_search import (
    freeze_selection,
    load_study_config,
    ranked_settings,
    select_best,
)
from music_recsys.listening_analysis import primary_rows
from music_recsys.search_analysis import (
    final_comparison,
    history_performance,
    parameter_sensitivity,
    plot_final,
    plot_performance,
    plot_search,
    plot_sensitivity,
    plot_tradeoffs,
    search_table,
    selected_curves,
    tradeoff_points,
    trial_curves,
    workflow_state,
)

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
saved_report_path = os.environ.get("MUSIC_RECSYS_REPORT_PATH")
DATASET_ID = os.environ.get(
    "MUSIC_RECSYS_DATASET_ID", "music4all-onion" if saved_report_path else "toy-music-v1"
)
TOP_K = int(os.environ.get("MUSIC_RECSYS_TOP_K", "10" if saved_report_path else "2"))
REPORT_PATHS = {
    "Popularity": Path(saved_report_path)
    if saved_report_path
    else ROOT / "examples/reports/toy-popularity.json"
}
MLFLOW_RUNS = {}
DETAILS_ROOT = None
BASELINE = "Popularity"
METRICS = ("ndcg", "recall")
DETAIL_METRIC = "recall"
BOOTSTRAP = BootstrapConfig(confidence=0.95, resamples=2000, seed=42)
HISTORY_EDGES = (0, 1, 5, 20)
USER_ATTRIBUTE = None


EXPERIMENT_CONFIG = Path(
    os.environ.get("MUSIC_RECSYS_EXPERIMENT_CONFIG", ROOT / "configs/workflow-toy.toml")
)
DIAGNOSTIC_TRIAL_ID = os.environ.get(
    "MUSIC_RECSYS_TRIAL_ID"
)  # 未指定は最初の成功試行。最良選択ではない
ACTION = None  # 明示操作: None / "select" / "freeze"
WORKFLOW = load_study_config(EXPERIMENT_CONFIG)
workflow = workflow_state(WORKFLOW)
REFERENCE_MODE = WORKFLOW.protocol == "binary-v1" and (
    bool(saved_report_path) or not (WORKFLOW.output / "study.json").exists()
)
experiment_settings = tomllib.loads(EXPERIMENT_CONFIG.read_text())
display(
    pd.DataFrame(
        [
            {
                "config": str(EXPERIMENT_CONFIG),
                "model": WORKFLOW.model,
                "protocol": WORKFLOW.protocol,
                "task": WORKFLOW.task,
                "window_days": WORKFLOW.evaluation.primary_window_days
                if WORKFLOW.evaluation
                else None,
                "day_weight": WORKFLOW.evaluation.primary_day_weight
                if WORKFLOW.evaluation
                else None,
                "validation_dataset": str(WORKFLOW.validation_dataset),
                "primary_metric": WORKFLOW.metric or "未指定：自動選択しません",
                "primary_k": WORKFLOW.primary_k,
                "generation_k": 20,
                "stage": WORKFLOW.stage,
                "seeds": WORKFLOW.seeds,
                "grid": WORKFLOW.grid,
                "smoke_users": WORKFLOW.smoke_users,
                "smoke_train_users": WORKFLOW.smoke_train_users,
                "max_trials": WORKFLOW.max_trials,
                "time_budget_seconds": WORKFLOW.time_budget_seconds,
            }
        ]
    )
)
display(workflow)
try:
    _, evaluation_conditions = experiment_context(WORKFLOW)
    display({"fixed_evaluation_conditions": evaluation_conditions})
except (ValueError, OSError) as error:
    print("条件の確認が必要:", error)

In [ ]:
if REFERENCE_MODE:
    sources = dict(REPORT_PATHS)
    if MLFLOW_RUNS:
        tracking = TrackingConfig.from_settings()
        for label, run_id in MLFLOW_RUNS.items():
            if label in sources:
                raise ValueError(f"duplicate report label: {label}")
            sources[label] = read_mlflow_report(
                run_id, tracking_uri=tracking.tracking_uri, details_root=DETAILS_ROOT
            )
    reports = load_reports(sources, dataset_id=DATASET_ID, top_k=TOP_K)
    conditions_df = pd.DataFrame(condition_rows(reports))
    display(
        conditions_df[["run", "model_name", "seed", "dataset_id", "top_k", "started_at", "details"]]
    )
    display(
        conditions_df[
            ["run", "dataset_sha256", "source_sha256", "config_sha256", "model_parameters"]
        ]
    )
    common_conditions = next(iter(reports.values()))["metadata"]["evaluation_conditions"]
    display(pd.DataFrame(common_conditions.items(), columns=["condition", "value"]))
    missing_details = [label for label, raw in reports.items() if "runs" not in raw]
    if missing_details:
        print("詳細なし（信頼区間・ユーザー別・失敗内訳は未分析）:", ", ".join(missing_details))
else:
    reports = {}
print("Next:", workflow["next_action"])

## 3. Search Results — Validationのみ

- Stage Aは1変数、Bは小規模Grid、Cは親探索の有望範囲を絞る。3変数以上はまず比較表を確認する。
- 1変数はseed平均の線、2変数はヒートマップ。失敗・未実行セルを0で埋めない。
- CLI: `python -m music_recsys.experiment_search --config <同じTOML> smoke` → `search`。


In [ ]:
search_df = pd.DataFrame()
if workflow["status"] != "mismatch":
    search_df = search_table(WORKFLOW)
if search_df.empty:
    print("探索結果は未取得。01の確認後、CLIでSmoke → searchを実行してください。")
else:
    display(search_df)
    search_fig = plot_search(WORKFLOW)
    if search_fig is not None:
        plt.show()
    elif len(WORKFLOW.grid) > 2:
        sensitivity_df = parameter_sensitivity(WORKFLOW)
        display(sensitivity_df)
        plot_sensitivity(
            sensitivity_df, metric=WORKFLOW.metric or "ndcg (diagnostic)", k=WORKFLOW.primary_k
        )
        plt.show()
        print(
            "3変数以上：他の条件を表のfixed_parametersに固定。図は先頭3変数、全変数の値は表で確認できます。"
        )
    else:
        print("描画できる探索変数・成功値がありません。Popularityは探索パラメータなしです。")
print("Next:", workflow["next_action"])

## 4. Model Selection — 最良・次点・全seed

- 主指標は開始前のTOML設定を使用。未指定・探索未完了・失敗・条件不一致なら確定しない。
- 同点はパラメータのcanonical表記で安定化。少差・範囲端・比較1件は「最適化完了」と解釈しない。
- `ACTION="select"`で選択artifactを保存し、03へ引き継ぐ。03で確認した後、`ACTION="freeze"`で固定する。


In [ ]:
selection_df = pd.DataFrame()
if ACTION is not None:
    if ACTION not in {"select", "freeze"}:
        raise ValueError("ACTIONはNone/select/freezeのみ。探索とTestはCLIで実行してください。")
    display({"explicit_action": ACTION, "config": str(EXPERIMENT_CONFIG), "phase": "validation"})
    display(select_best(WORKFLOW) if ACTION == "select" else freeze_selection(WORKFLOW))
    workflow = workflow_state(WORKFLOW)
if (
    workflow["status"]
    not in {"mismatch", "not_run", "smoke_passed", "incomplete", "failed", "running", "interrupted"}
    and WORKFLOW.metric
):
    selection_df = pd.DataFrame(ranked_settings(WORKFLOW))
    display(selection_df.head(10))
for warning in workflow.get("warnings", []):
    print("確認:", warning)
print("Next:", workflow["next_action"])

## 5. Performance Analysis — Validationの感度・トレードオフ

- 選択済み設定の同じ上位20曲からK=1〜20を評価。K=10を主設定とする。
- shaded areaはユーザー単位の95%区間。seed間のばらつきはModel Selectionの`seed_std`と区別する。
- Gradedは関連度未確定、特徴のないILD、未計測のTokens / Cost / Calls / Memoryは未測定。代替値や0を入れない。


In [ ]:
workflow_curves = pd.DataFrame()
if workflow["status"] in {
    "review_pending",
    "review_rejected",
    "freeze_pending",
    "test_ready",
    "test_succeeded",
    "test_failed",
}:
    workflow_curves = selected_curves(WORKFLOW, resamples=BOOTSTRAP.resamples)
    display(primary_rows(workflow_curves)[lambda rows: rows.k.isin([5, 10, 20])])
    plot_performance(workflow_curves)
    plt.show()
    tradeoff_df = tradeoff_points(WORKFLOW)
    display(tradeoff_df)
    plot_tradeoffs(tradeoff_df)
    plt.show()
    if WORKFLOW.protocol == "listening-v2":
        print("多様性は主タスクの候補で計測。ILDは特徴の揃う有効曲2曲以上。CIは主期間・主λのみ。")
    display(history_performance(WORKFLOW))
    print("観察：Kによる変化、精度と多様性・人気度の関係、履歴量別の対象人数を確認してください。")
elif WORKFLOW.metric is None and not search_df.empty and (search_df.status == "succeeded").any():
    successful = search_df[search_df.status == "succeeded"]
    diagnostic_id = DIAGNOSTIC_TRIAL_ID or successful.iloc[0].trial_id
    print("診断表示：指定した1試行。最良設定の選択ではありません。trial_id:", diagnostic_id)
    workflow_curves = trial_curves(WORKFLOW, diagnostic_id, resamples=BOOTSTRAP.resamples)
    display(primary_rows(workflow_curves)[lambda rows: rows.k.isin([5, 10, 20])])
    plot_performance(workflow_curves)
    plt.show()
    tradeoff_df = tradeoff_points(WORKFLOW)
    display(tradeoff_df)
    plot_tradeoffs(tradeoff_df)
    plt.show()
    print("主タスク・主指標を事前指定した別実験を用意するまで、選択・固定・Testには進めません。")
else:
    print("保存済みの同条件データが必要です。02のModel Selectionで状態を確認してください。")
if REFERENCE_MODE:
    print("以下は探索とは別の保存済み参考結果です。条件を混ぜて比較しないでください。")

### 保存済み参考結果：Performance

- 平均は評価対象の要求を等重みで集計。正解なしは除外し、失敗は正解があれば0として含める。
- 信頼区間：ユーザー単位のpercentile bootstrap。複数要求・会話ターンはユーザーごとまとめて再標本化。
- ユーザー2人未満・詳細結果なしでは信頼区間を表示しない。
- 合成データは少数ユーザーの動作確認。信頼区間を実データでの性能保証と解釈しない。

In [ ]:
if REFERENCE_MODE:
    performance_df = pd.DataFrame(
        [
            row
            for metric in METRICS
            for row in performance_rows(reports, metric=metric, bootstrap=BOOTSTRAP)
        ]
    )
    display(performance_df)

    fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 3.5), squeeze=False)
    for ax, metric in zip(axes.flat, METRICS):
        frame = performance_df[performance_df["metric"] == metric].reset_index(drop=True)
        valid = frame[frame["mean"].notna()]
        ax.scatter(valid.index, valid["mean"], color="#267d85", label="Request mean")
        intervals = frame.dropna(subset=["ci_low", "ci_high"])
        ax.vlines(intervals.index, intervals["ci_low"], intervals["ci_high"], color="#267d85")
        ax.set_xticks(frame.index, frame["run"], rotation=20)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"{metric.upper()}@{TOP_K} / {BOOTSTRAP.confidence:.0%} user CI")
        if valid.empty:
            ax.text(0.5, 0.5, "Undefined", ha="center", transform=ax.transAxes)
    fig.tight_layout()
    plt.show()

### 基準モデルとの差・反復実行のばらつき

- 対応差分：同じ要求IDで対応付け、`比較run − BASELINE` を計算。
- 差分の信頼区間もユーザー単位。複数runを指定したときに表示する。
- 反復表はモデル名・設定・実装コードが一致するrunを集計。異なるseedと同じseedの再実行を件数で区別。
- `run_std` は実行間の標準偏差。ユーザー間のばらつき・信頼区間とは別に扱う。

In [ ]:
if REFERENCE_MODE:
    difference_df = pd.DataFrame(
        paired_differences(reports, baseline=BASELINE, metric=DETAIL_METRIC, bootstrap=BOOTSTRAP)
    )
    if difference_df.empty:
        print("比較runは1件。別の保存結果をSetupに追加すると対応差分を表示します。")
    else:
        display(difference_df)
    repetition_df = pd.DataFrame(repetition_rows(reports, metric=DETAIL_METRIC))
    display(repetition_df)

### 保存済み参考結果：Detailed Analysis

### ユーザー別・履歴量別・属性別

- ユーザー別の平均・分布と、履歴量グループの性能を確認する。
- 履歴量は推薦時点の入力から取得。区間は左端を含み、右端を含まない。
- 属性は保存済み `user_attributes` のみ使用。未取得は `(missing)` に分類。
- グループごとに要求数・評価対象数・ユーザー数を表示。少数群の解釈に注意する。

In [ ]:
if REFERENCE_MODE:
    users_df = pd.DataFrame(user_rows(reports, metric=DETAIL_METRIC, baseline=BASELINE))
    history_df = pd.DataFrame(
        group_rows(reports, metric=DETAIL_METRIC, history_edges=HISTORY_EDGES, bootstrap=BOOTSTRAP)
    )
    if users_df.empty:
        print("詳細結果なし：ユーザー別・履歴量別の分析は未実施。")
    else:
        display(users_df.pivot(index="user", columns="run", values="mean"))
        display(users_df.pivot(index="user", columns="run", values="delta_from_baseline"))
        display(history_df)
        fig, ax = plt.subplots(figsize=(8, 3.5))
        groups = [
            (label, frame["mean"].dropna().tolist())
            for label, frame in users_df.groupby("run", sort=False)
        ]
        groups = [(label, values) for label, values in groups if values]
        if groups:
            ax.boxplot([values for _, values in groups], tick_labels=[label for label, _ in groups])
        else:
            ax.text(0.5, 0.5, "No evaluated users", ha="center", transform=ax.transAxes)
        ax.set_ylabel(f"User mean {DETAIL_METRIC}@{TOP_K}")
        ax.set_ylim(-0.05, 1.05)
        fig.tight_layout()
        plt.show()

    if USER_ATTRIBUTE is None:
        print("ユーザー属性は未選択。既定の合成結果には属性データがありません。")
    else:
        display(
            pd.DataFrame(
                group_rows(
                    reports, metric=DETAIL_METRIC, attribute=USER_ATTRIBUTE, bootstrap=BOOTSTRAP
                )
            )
        )

### Diversity・Coverage・Novelty

| 指標 | 比較時に固定する定義 |
| --- | --- |
| Diversity | 類似度・特徴量・Embeddingの版、集計方法 |
| Coverage | 対象カタログ・評価対象集合・K |
| Novelty | 人気度の参照期間・算出方法 |

- `additional_metrics` に保存された値と定義を表示。定義不一致は比較不可。
- 現在は未計算。Notebook内で指標を作らず、未測定を0に置き換えない。

In [ ]:
if REFERENCE_MODE:
    additional_df = pd.DataFrame(additional_metric_rows(reports))
    display(additional_df)
    measured = additional_df.dropna(subset=["value"])
    if not measured.empty:
        for metric, frame in measured.groupby("metric", sort=False):
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.bar(frame["run"], frame["value"], color="#627d98")
            ax.set_title(metric)
            fig.tight_layout()
            plt.show()

### 失敗分析

| 分類 | 内容 |
| --- | --- |
| `error` / `error:種類` | 推薦処理の失敗・例外の種類 |
| `short_output` | 有効な推薦曲がK未満 |
| `invalid_items` / `duplicate_items` | 無効な曲・重複曲を含む要求 |
| `no_relevant_candidates` | 候補内に正解なし。評価対象外 |
| `zero_recall` | 評価対象だが正解曲を推薦できなかった要求 |

- 分類は重複するため合計しない。出力不足は候補不足の場合も含む。

In [ ]:
if REFERENCE_MODE:
    failure_summary_df = pd.DataFrame(
        [
            {
                "run": label,
                **{
                    key: raw["summary"][key]
                    for key in ("n_requests", "n_failed", "n_short", "n_excluded")
                },
            }
            for label, raw in reports.items()
        ]
    )
    display(failure_summary_df)
    failures_df = pd.DataFrame(failure_rows(reports))
    if failures_df.empty:
        print("詳細結果なし：失敗の内訳は未分析。集計済み件数は上の表で確認できます。")
    else:
        display(failures_df)
        ax = failures_df.pivot(index="category", columns="run", values="count").plot.barh(
            figsize=(9, 4)
        )
        ax.set_xlabel("Requests (overlapping categories)")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()

### 保存済み参考結果：Efficiency

- 学習時間、推論時間の平均・中央値・p95、推論コストを比較。
- コストは全要求分の金額・通貨・算定基準が揃う場合のみ合計する。欠測を無料と扱わない。
- 実行環境・並列数・キャッシュ等の条件を確認する。未記録の条件は推測しない。
- 散布図は性能との関係を探索するもの。合成データの短い計測から速度の優劣を結論付けない。

In [ ]:
if REFERENCE_MODE:
    efficiency_df = pd.DataFrame(efficiency_rows(reports))
    display(efficiency_df)
    tradeoff_df = performance_df[performance_df["metric"] == DETAIL_METRIC].merge(
        efficiency_df, on="run"
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    latency = tradeoff_df.dropna(subset=["latency_mean", "mean"])
    axes[0].scatter(latency["latency_mean"], latency["mean"], color="#267d85")
    for _, row in latency.iterrows():
        axes[0].annotate(row["run"], (row["latency_mean"], row["mean"]))
    axes[0].set_xlabel("Mean inference time (s / request)")
    axes[0].set_ylabel(f"{DETAIL_METRIC}@{TOP_K}")
    axes[0].set_ylim(-0.05, 1.05)

    costs = tradeoff_df.dropna(subset=["cost_per_request", "mean"])
    units = costs[["cost_currency", "cost_basis"]].drop_duplicates()
    if costs.empty:
        axes[1].text(0.5, 0.5, "Cost not measured", ha="center", transform=axes[1].transAxes)
    elif len(units) != 1:
        axes[1].text(
            0.5, 0.5, "Different cost units / bases", ha="center", transform=axes[1].transAxes
        )
    else:
        axes[1].scatter(costs["cost_per_request"], costs["mean"], color="#b77742")
        for _, row in costs.iterrows():
            axes[1].annotate(row["run"], (row["cost_per_request"], row["mean"]))
        axes[1].set_xlabel(f"Cost ({units.iloc[0]['cost_currency']} / request)")
    axes[1].set_ylabel(f"{DETAIL_METRIC}@{TOP_K}")
    axes[1].set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plt.show()

## 6. Final Comparison — 固定後のTestだけ

- `freeze`後、CLIの`test`で固定パラメータ・全seedをTrain＋Validationで再学習する。
- モデル間比較の設定TOMLは`[comparison].configs`へ追加。同じTest hash・候補・評価条件だけを比較する。
- Testの結果から設定を変更しない。Test未実行・失敗モデルは状態表で区別する。


In [ ]:
comparison_paths = [WORKFLOW.path] + [
    (WORKFLOW.path.parent / p).resolve()
    for p in experiment_settings.get("comparison", {}).get("configs", [])
]
comparison_configs = [load_study_config(p) for p in dict.fromkeys(comparison_paths)]
display(pd.DataFrame([{"experiment": c.name, **workflow_state(c)} for c in comparison_configs]))
final_df = pd.DataFrame()
try:
    final_df = final_comparison(comparison_configs)
    if final_df.empty:
        print("Testは未実行。03で確認 → 設定固定 → CLI testの順に進めてください。")
    else:
        display(primary_rows(final_df)[lambda rows: rows.k.isin([5, 10, 20])])
        plot_final(final_df)
        plt.show()
        print("Testの比較条件を統一。探索回数・総時間・seed数も併せて報告してください。")
except ValueError as error:
    print("最終比較を停止:", error)

## 7. Summary / Next Actions

| 状況 | 次に行うこと | 戻り先 |
| --- | --- | --- |
| 正解保有ユーザーが少ない | データ・分割・対象人数を確認 | 01_data |
| Smoke / モデル失敗 | error_typeとモデル実装を確認 | src・CLI |
| 予算切れ・探索未完了 | 予算・範囲を確認してsearch再開 | Setup・CLI |
| 最良が範囲端・差が小さい | 範囲拡張、コスト・seed差の比較 | Search Results |
| 選択済み・確認前 | 同じ選択artifactの推薦を確認 | 03_recommendation |
| 03確認済み | freezeし、固定条件でtest | Model Selection・CLI |
| Test成功 | 同条件モデルの最終比較、限界を記述 | Final Comparison |

- 所見・失敗・次の仮説を記録する。判断不能を成功として扱わない。
- [検証フローとCLI](../docs/experiment-workflow.md) / [01_data](01_data.ipynb) / [03_recommendation](03_recommendation.ipynb)
